# Data Preprocessing

This notebook preprocesses the data from Davidson et al. (2017) and pickles it for future use.

It pickles two datasets. One with the original hate speech class, and another with the breakdown of the hate class into directed and generalized hate speech.


In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from utils import preprocess, create_lookup_tables, create_pad_fn, pad_tweets,\
                  hate_classification, change_hate_labels

In [ ]:
df = pd.read_csv("./data/labeled_data.csv", index_col=0)
raw_tweets = df.tweet
raw_labels = df["class"].values

In [ ]:
df.head()

### Data Cleaning 

In [ ]:
tweets = raw_tweets.map(preprocess)
print("Example of a raw tweet:\n{}".format(raw_tweets[68]))
print("\nIts cleaned version is:\n{}".format(preprocess(raw_tweets[68])))

### Checking for outliers

In [ ]:
# Get cleaned tweets
df["clean_tweet"] = tweets

# Get their word count
df["word_count"] = df.clean_tweet.apply(lambda x : len(x.split()))

df.word_count.describe()

In [ ]:
# Check tweets with the minimum word count
df.loc[df.word_count == df.word_count.min(),]

Looks good. Let's check the tweet(s) with the maximum word count.

In [ ]:
# Check tweets with the maximum length
df.loc[df.word_count == df.word_count.max(),]

There's something strange going on. Let's check the tweet again.

In [ ]:
df.loc[df.word_count == df.word_count.max(),].tweet.values

The tweet contains a lot of new lines. It's hard to know why, but I'll choose to remove them.

In [ ]:
old_tweet = df.loc[df.word_count == df.word_count.max(),].tweet.values[0]
new_tweet = old_tweet[:old_tweet.find("\r")]
df.loc[df.word_count == df.word_count.max(), "tweet"] = new_tweet
df.loc[df.word_count == df.word_count.max(), "clean_tweet"] = preprocess(new_tweet)
df.loc[df.word_count == df.word_count.max(), "word_count"] = len(preprocess(new_tweet).split())

Let's check it again.

In [ ]:
# Check tweets with the maximum length
df.loc[df.word_count == df.word_count.max(),]

In [ ]:
df.loc[df.word_count == df.word_count.max(),].clean_tweet.values[0]

Sighes. Well, format-wise it is okay.

### Create lookup tables

In [ ]:
vocab_to_int, int_to_vocab = create_lookup_tables(tweets)

In [ ]:
print("The size of the vocabulary is: {} tokens.".format(len(vocab_to_int)))
vocab = list(vocab_to_int.keys())
np.random.shuffle(vocab)
print("These are 10 randomly sample words in the vocabulary:\n{}".format(vocab[:10]))
del vocab

###  Padding the Data

In [ ]:
# Hemker sets the sentence length to 100. We'll do the same here for consistency, but
# note that the max sentence length in our dataset is actually 91.
MAX_LENGTH = 100  # df.word_count.max()
pad_tweets = create_pad_fn(MAX_LENGTH)
df["padded_tweets"] = df.clean_tweet.map(pad_tweets)
print(df.padded_tweets[10])

### Tokenizing the Data

In [ ]:
tweets_ints = np.array([[vocab_to_int[word] for word in tweet.split()] for tweet in df.padded_tweets.values])
print(tweets_ints[10])

### Hate Subclass Extraction

In [ ]:
hate_tweets = tweets[df["class"] == 0].values
_hate_prnt = lambda x : "Generalized" if hate_classification(x) == 4 else "Directed"

print("Example of a hateful tweet: \n{}".format(hate_tweets[20]))
print("Its type of hate speech is: {}\n".format(_hate_prnt(hate_tweets[20])))

print("Example of a hateful tweet:\n{}".format(hate_tweets[10]))
print("Its type of hate speech is: {}\n".format(_hate_prnt(hate_tweets[10])))

### Change hate labels

In [ ]:
labels = change_hate_labels(tweets, raw_labels)
pd.Series(labels).value_counts()

### Save Files

In [ ]:
np.save("data/tweets", tweets_ints)
np.save("data/hate_original", raw_labels)
np.save("data/hate_breakdown", labels)

In [ ]:
with open('vocab_to_int.json', 'w') as f:
    json.dump(vocab_to_int, f)
    
with open('int_to_vocab.json', 'w') as f:
    json.dump(int_to_vocab, f)